# 09 — Bronze: Receita Federal — Estabelecimentos

## Purpose
Ingests Receita Federal Estabelecimentos data (10 shards) into the Bronze layer.
Each shard has ~5M rows — total ~50M rows across all shards.

## Input
- `data/raw/receita_federal/{YYYY-MM}/Estabelecimentos{0-9}.zip` — 10 shards per month
- Shard 0 alone: ~2 GB compressed, ~5M rows

## Output
- `data/bronze/rf_estabelecimentos/_year_month={YYYY-MM}/data.parquet` — single file per month
- Schema: 30 source columns (all STRING) + 5 technical columns

## Steps
1. Imports and configuration
2. Schema definition
3. Process shards
4. Validation
5. Ops registration

## Notes
- Largest source in the project — ~50M rows, ~20 GB compressed total
- Each shard extracted to temp file → DuckDB reads CSV → temp deleted (avoids RAM)
- Shards appended to single Parquet via atomic os.replace()
- Encoding: latin-1 (RF standard) with ignore_errors=true as safety net
- Silver: CONCAT(cnpj_basico, cnpj_ordem, cnpj_dv) → cnpj_raw (14 digits)
- Idempotent — deletes existing output before processing shards
- ADR-002: all source columns as STRING


## Step 1 — Imports and configuration

In [ ]:
import json
import os
import sys
import tempfile
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import get_project_root, to_sql_path
from duckdb_utils import get_connection
from validation import CheckSuite
from bootstrap_log import append_log, make_entry
from pipeline import check_landing_gate

# ---------------------------------------------------------------------------
# Project root and paths
# ---------------------------------------------------------------------------
PROJECT_ROOT = get_project_root()

RF_ROOT     = PROJECT_ROOT / "data" / "raw"    / "receita_federal"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze" / "rf_estabelecimentos"
DRIFT_LOG   = PROJECT_ROOT / "db"  / "schema_drift_log.json"
LOG_PATH    = PROJECT_ROOT / "db"  / "bootstrap_log.json"

# ---------------------------------------------------------------------------
# Landing gate check
# ---------------------------------------------------------------------------
check_landing_gate(PROJECT_ROOT)

# ---------------------------------------------------------------------------
# Month selection
# ---------------------------------------------------------------------------
available_months = sorted([d.name for d in RF_ROOT.iterdir() if d.is_dir()])
if not available_months:
    raise FileNotFoundError(
        f"No month directories found in {RF_ROOT}\n"
        "Run 01_bootstrap_receita_federal.py first."
    )

SAMPLE_MONTH = available_months[-1]
SAMPLE_DIR   = RF_ROOT / SAMPLE_MONTH

# ---------------------------------------------------------------------------
# Pipeline metadata
# ---------------------------------------------------------------------------
SOURCE_ID  = "receita_federal"
BATCH_ID   = str(uuid.uuid4())
STARTED_AT = datetime.now(timezone.utc).isoformat()  # string from the start

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"SAMPLE_MONTH : {SAMPLE_MONTH}")
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"BATCH_ID     : {BATCH_ID}")
print(f"STARTED_AT   : {STARTED_AT}")
print()
print("[WARNING] This notebook processes ~50M rows across 10 shards.")
print("          Estimated runtime: 20-40 min depending on hardware.")
print("          Each shard extracts to a temp CSV — ensure ~20 GB free disk space.")


## Step 2 — Schema definition

In [ ]:
# ---------------------------------------------------------------------------
# Estabelecimentos layout — 30 columns per official RF documentation
# Reference: https://arquivos.receitafederal.gov.br (layout PDF)
# Columns identified by position — CSV has NO headers
# ---------------------------------------------------------------------------
ESTAB_COLS = [
    "cnpj_basico",               #  0 — 8-digit root
    "cnpj_ordem",                #  1 — 4-digit order
    "cnpj_dv",                   #  2 — 2-digit check digits → Silver: CONCAT → cnpj_raw
    "identificador_matriz_filial", # 3 — 1=matriz, 2=filial
    "nome_fantasia",             #  4
    "situacao_cadastral",        #  5 — 01=nula,2=ativa,3=suspensa,4=inapta,8=baixada
    "data_situacao_cadastral",   #  6 — YYYYMMDD (sentinel: 00000000)
    "motivo_situacao_cadastral", #  7 — code (FK rf_motivos)
    "nome_cidade_exterior",      #  8 — foreign establishments only
    "pais",                      #  9 — code (FK rf_paises)
    "data_inicio_atividade",     # 10 — YYYYMMDD (sentinel: 00000000)
    "cnae_fiscal_principal",     # 11 — 7-digit CNAE code (FK rf_cnaes)
    "cnae_fiscal_secundaria",    # 12 — semicolon-separated list of CNAE codes
    "tipo_logradouro",           # 13
    "logradouro",                # 14
    "numero",                    # 15
    "complemento",               # 16
    "bairro",                    # 17
    "cep",                       # 18 — 8-digit CEP without dash
    "uf",                        # 19 — 2-letter state code
    "municipio",                 # 20 — code (FK rf_municipios)
    "ddd_1",                     # 21
    "telefone_1",                # 22
    "ddd_2",                     # 23
    "telefone_2",                # 24
    "ddd_fax",                   # 25
    "fax",                       # 26
    "correio_eletronico",        # 27
    "situacao_especial",         # 28
    "data_situacao_especial",    # 29 — YYYYMMDD (sentinel: 00000000)
]

EXPECTED_COL_COUNT = len(ESTAB_COLS)  # drift = any shard with != 30 columns

TECHNICAL_COLUMNS = [
    "_source_id",
    "_batch_id",
    "_ingested_at",
    "_source_file",
    "_year_month",
]

ALL_COLUMNS = ESTAB_COLS + TECHNICAL_COLUMNS

# DuckDB zero-pads column names when ncols >= 10: column00, column01, ... column29
# Built at module level — reused in every shard write
COL_ALIASES = ", ".join(f"column{i:02d} AS {c}" for i, c in enumerate(ESTAB_COLS))

print(f"Source columns    : {len(ESTAB_COLS)}")
print(f"Technical columns : {len(TECHNICAL_COLUMNS)}")
print(f"Total columns     : {len(ALL_COLUMNS)}")
print(f"Expected per shard: {EXPECTED_COL_COUNT} source columns")


## Step 3 — Process shards

In [ ]:
def _log_drift_event(shard_name: str, actual_cols: int, expected_cols: int) -> None:
    """
    Append a schema drift event to schema_drift_log.json.

    For positional CSV sources, drift = unexpected column count.

    Parameters
    ----------
    shard_name    : filename of the affected shard
    actual_cols   : number of columns found
    expected_cols : expected column count (EXPECTED_COL_COUNT)
    """
    event = {
        "batch_id"      : BATCH_ID,
        "source_id"     : SOURCE_ID,
        "object"        : "bronze_rf_estabelecimentos",
        "event_type"    : "SCHEMA_DRIFT",
        "severity"      : "WARN",
        "action"        : "CONTINUE",
        "shard"         : shard_name,
        "expected_cols" : expected_cols,
        "actual_cols"   : actual_cols,
        "logged_at"     : datetime.now(timezone.utc).isoformat(),
    }
    existing = []
    if DRIFT_LOG.exists():
        with open(DRIFT_LOG, "r", encoding="utf-8") as f:
            try:
                existing = json.load(f)
            except json.JSONDecodeError:
                existing = []
    existing.append(event)
    DRIFT_LOG.parent.mkdir(parents=True, exist_ok=True)
    with open(DRIFT_LOG, "w", encoding="utf-8") as f:
        json.dump(existing, f, ensure_ascii=False, indent=2)


# ---------------------------------------------------------------------------
# Discover shards
# ---------------------------------------------------------------------------
shard_files = sorted(SAMPLE_DIR.glob("Estabelecimentos*.zip"))

if not shard_files:
    raise FileNotFoundError(
        f"No Estabelecimentos*.zip files found in {SAMPLE_DIR}\n"
        "Run 01_bootstrap_receita_federal.py first."
    )

print(f"Shards found : {len(shard_files)}")
for s in shard_files:
    print(f"  {s.name} — {s.stat().st_size / 1e6:.1f} MB compressed")
print()

# ---------------------------------------------------------------------------
# Process shards — extract to temp → sanitize → DuckDB reads → append to Parquet
# ---------------------------------------------------------------------------
BRONZE_ROOT.mkdir(parents=True, exist_ok=True)
partition_dir = BRONZE_ROOT / f"_year_month={SAMPLE_MONTH}"
partition_dir.mkdir(parents=True, exist_ok=True)
output_file  = partition_dir / "data.parquet"
output_path  = to_sql_path(output_file)

# Idempotency: remove existing output before re-processing all shards
if output_file.exists():
    output_file.unlink()
    print(f"Removed existing: {output_file.name}")

con = get_connection()  # in-memory — Parquet I/O does not require persistent DB

# C1 control bytes (0x80–0x9F) are invalid in strict latin-1 (ISO-8859-1).
# RF Estabelecimentos files occasionally contain 0x8F bytes — valid in
# Windows-1252 but rejected by DuckDB's latin-1 validator.
# These are non-printable C1 control characters with no semantic value.
# Strategy: replace with space (0x20) during extraction — preserves row structure.
# ignore_errors=true does NOT suppress encoding errors — only parse errors.
INVALID_C1  = set(range(0x80, 0xA0))
CHUNK_SIZE  = 64 * 1024 * 1024  # 64 MB

results       = []
total_records = 0
has_drift     = False
error_count   = 0

for i, shard_path in enumerate(shard_files, 1):
    tmp_path = None
    try:
        print(f"[{i:02d}/{len(shard_files)}] Extracting {shard_path.name}...")

        # Step 1 — Extract ZIP to temp file via streaming chunks
        with zipfile.ZipFile(shard_path) as zf:
            inner_name = zf.namelist()[0]
            with zf.open(inner_name) as zf_stream, \
                 tempfile.NamedTemporaryFile(
                     suffix=".csv", delete=False, mode="wb"
                 ) as tmp:
                while True:
                    chunk = zf_stream.read(CHUNK_SIZE)
                    if not chunk:
                        break
                    tmp.write(chunk)
                tmp.flush()
                os.fsync(tmp.fileno())
                tmp_path = tmp.name

        # Step 2 — Sanitize C1 control bytes invalid in latin-1
        # Rewrites temp file in-place replacing 0x80-0x9F with space
        sanitized_path = tmp_path + ".clean"
        with open(tmp_path, "rb") as f_in, \
             open(sanitized_path, "wb") as f_out:
            while True:
                chunk = f_in.read(CHUNK_SIZE)
                if not chunk:
                    break
                f_out.write(
                    bytes(b if b not in INVALID_C1 else 0x20 for b in chunk)
                )
            f_out.flush()
            os.fsync(f_out.fileno())
        os.unlink(tmp_path)
        tmp_path = sanitized_path

        tmp_posix  = to_sql_path(tmp_path)
        source_sql = to_sql_path(shard_path)

        # Schema drift detection — sample first row to count columns
        sample_desc = con.execute(f"""
            SELECT * FROM read_csv('{tmp_posix}',
                sep=';', header=false,
                encoding='latin-1',
                ignore_errors=true,
                sample_size=100
            ) LIMIT 1
        """).description
        actual_cols = len(sample_desc)

        if actual_cols != EXPECTED_COL_COUNT:
            has_drift = True
            _log_drift_event(shard_path.name, actual_cols, EXPECTED_COL_COUNT)
            print(f"  [DRIFT] Expected {EXPECTED_COL_COUNT} cols, got {actual_cols}")

        if i == 1:
            # First shard: CREATE the Parquet file
            con.execute(f"""
                COPY (
                    SELECT
                        {COL_ALIASES},
                        '{SOURCE_ID}'     AS _source_id,
                        '{BATCH_ID}'      AS _batch_id,
                        CURRENT_TIMESTAMP AS _ingested_at,
                        '{source_sql}'    AS _source_file,
                        '{SAMPLE_MONTH}'  AS _year_month
                    FROM read_csv('{tmp_posix}',
                        sep=';', header=false,
                        encoding='latin-1',
                        ignore_errors=true
                    )
                ) TO '{output_path}'
                (FORMAT PARQUET, COMPRESSION SNAPPY)
            """)
        else:
            # Guard — only UNION if previous shards succeeded
            if not output_file.exists():
                raise RuntimeError(
                    f"Output Parquet not found — previous shard failed. "
                    "Re-run from the beginning of this cell."
                )
            tmp_out = output_path + ".tmp"
            con.execute(f"""
                COPY (
                    SELECT * FROM read_parquet('{output_path}')
                    UNION ALL
                    SELECT
                        {COL_ALIASES},
                        '{SOURCE_ID}'     AS _source_id,
                        '{BATCH_ID}'      AS _batch_id,
                        CURRENT_TIMESTAMP AS _ingested_at,
                        '{source_sql}'    AS _source_file,
                        '{SAMPLE_MONTH}'  AS _year_month
                    FROM read_csv('{tmp_posix}',
                        sep=';', header=false,
                        encoding='latin-1',
                        ignore_errors=true
                    )
                ) TO '{tmp_out}'
                (FORMAT PARQUET, COMPRESSION SNAPPY)
            """)
            os.replace(tmp_out, output_path)

        current_total = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{output_path}')"
        ).fetchone()[0]
        shard_count   = current_total - total_records
        total_records = current_total

        results.append({"shard": shard_path.name, "status": "SUCCESS",
                         "records": shard_count, "error": None})
        print(f"  OK {shard_count:>8,} rows — cumulative: {total_records:>12,}")

    except Exception as e:
        error_count += 1
        results.append({"shard": shard_path.name, "status": "ERROR",
                         "records": 0, "error": str(e)})
        print(f"  ERR {str(e)[:100]}")

    finally:
        # Always delete temp file — even if an exception occurs
        if tmp_path and Path(tmp_path).exists():
            os.unlink(tmp_path)

con.close()

print()
print(f"Total records : {total_records:,}")
print(f"Shards OK     : {len(shard_files) - error_count}/{len(shard_files)}")
print(f"Errors        : {error_count}")
print(f"Schema drift  : {has_drift}")

## Step 4 — Validation

In [ ]:
suite   = CheckSuite("bronze_rf_estabelecimentos")
con_val = get_connection()

# Check 1 — no shard errors
suite.add(
    "No shard processing errors",
    error_count == 0,
    f"{error_count} error(s)",
)

# Check 2 — output Parquet exists
suite.add(
    "Output Parquet file exists",
    output_file.exists(),
    str(output_file),
)

# Check 3 — total records > 0
suite.add(
    "Total records written > 0",
    total_records > 0,
    f"{total_records:,}",
)

# Check 4 — record count matches sum of shard counts
bronze_total = con_val.execute(
    f"SELECT COUNT(*) FROM read_parquet('{output_path}')"
).fetchone()[0]
suite.add(
    "Record count matches shard sum",
    bronze_total == total_records,
    f"parquet={bronze_total:,}  sum={total_records:,}",
)

# Check 5 — column count correct
actual_cols = con_val.execute(
    f"SELECT COUNT(*) FROM (DESCRIBE SELECT * FROM read_parquet('{output_path}') LIMIT 0)"
).fetchone()[0]
suite.add(
    "Column count correct",
    actual_cols == len(ALL_COLUMNS),
    f"found={actual_cols}  expected={len(ALL_COLUMNS)}",
)

# Check 6 — all technical columns present
existing_cols = {
    row[0] for row in con_val.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{output_path}') LIMIT 0"
    ).fetchall()
}
missing_tech = [c for c in TECHNICAL_COLUMNS if c not in existing_cols]
suite.add(
    "All technical columns present",
    not missing_tech,
    f"missing: {missing_tech}" if missing_tech else "all present",
)

# Check 7 — cnpj components are STRING (not INTEGER)
cnpj_types = con_val.execute(f"""
    SELECT column_name, column_type
    FROM (DESCRIBE SELECT * FROM read_parquet('{output_path}') LIMIT 0)
    WHERE column_name IN ('cnpj_basico', 'cnpj_ordem', 'cnpj_dv')
""").fetchall()
all_varchar = all(row[1] == "VARCHAR" for row in cnpj_types)
suite.add(
    "cnpj components are VARCHAR (not INTEGER)",
    all_varchar,
    str({row[0]: row[1] for row in cnpj_types}),
)

# Check 8 — no zero-record shards
zero_shards = [r["shard"] for r in results if r["status"] == "SUCCESS" and r["records"] == 0]
suite.add(
    "No zero-record shards",
    not zero_shards,
    f"empty: {zero_shards}" if zero_shards else "all shards contributed records",
)

# Check 9 — schema drift monitored (always PASS)
suite.add(
    "Schema drift monitored",
    True,
    "drift detected — check schema_drift_log.json" if has_drift else "none detected",
)

con_val.close()  # close before potential raise
suite.report()
suite.assert_all_pass()


## Step 5 — Ops registration

In [ ]:
bytes_written = output_file.stat().st_size if output_file.exists() else 0
errors_detail = (
    "; ".join(f"{r['shard']}: {r['error'][:60]}" for r in results if r["status"] == "ERROR")
    if error_count > 0 else None
)

entry = make_entry(
    source_id     = SOURCE_ID,
    period        = SAMPLE_MONTH,
    status        = "SUCCESS" if error_count == 0 else "PARTIAL",
    records       = total_records,
    bytes_written = bytes_written,
    started_at    = STARTED_AT,
    finished_at   = datetime.now(timezone.utc).isoformat(),
    error_message = errors_detail,
    # Bronze-specific extra fields
    batch_id         = BATCH_ID,
    layer            = "bronze",
    object           = "rf_estabelecimentos",
    files            = len(shard_files),
    has_rescued_data = has_drift,
    drift_months     = 1 if has_drift else 0,
)

append_log(LOG_PATH, entry)

print(f"Batch registered   : {BATCH_ID}")
print(f"Status             : {entry['status']}")
print(f"Records            : {total_records:,}")
print(f"Bytes written      : {bytes_written / (1024*1024):.1f} MB")
print(f"Has rescued data   : {has_drift}")
print(f"Log                : {LOG_PATH}")
